<a href="https://colab.research.google.com/github/shivani11-glitch/seizure-detection/blob/main/week1_confidence_heads.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 1 — Confidence Heads


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## ConfidenceHeads Class

In [ ]:
class ConfidenceHeads(nn.Module):
    def __init__(self):
        super(ConfidenceHeads, self).__init__()
        self.eeg_head = nn.Sequential(
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
        self.ecg_head = nn.Sequential(
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
        self.semg_head = nn.Sequential(
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
        self.fusion_weights = nn.Parameter(torch.tensor([0.6, 0.2, 0.2]))

    def forward(self, x):
        pooled = torch.mean(x, dim=1)
        eeg_conf = self.eeg_head(pooled)
        ecg_conf = self.ecg_head(pooled)
        semg_conf = self.semg_head(pooled)
        weights = F.softmax(self.fusion_weights, dim=0)
        fused_conf = weights[0] * eeg_conf + weights[1] * ecg_conf + weights[2] * semg_conf
        return eeg_conf, ecg_conf, semg_conf, fused_conf

## Thresholds and apply_thresholds

In [ ]:
SEIZURE_THRESHOLD = 0.5
SUDEP_THRESHOLD = 0.75

def apply_thresholds(eeg_conf: float, ecg_conf: float, semg_conf: float, fused_conf: float) -> dict:
    seizure_detected = fused_conf >= SEIZURE_THRESHOLD
    sudep_risk = fused_conf >= SUDEP_THRESHOLD
    return {
        "eeg_confidence": float(eeg_conf),
        "ecg_confidence": float(ecg_conf),
        "semg_confidence": float(semg_conf),
        "fused_confidence": float(fused_conf),
        "seizure_detected": bool(seizure_detected),
        "sudep_risk": bool(sudep_risk)
    }

## Sanity Check

In [ ]:
model = ConfidenceHeads()
fake_input = torch.randn(8, 32, 128)
eeg_conf, ecg_conf, semg_conf, fused_conf = model(fake_input)

print(f"Sample EEG confidence: {eeg_conf[0].item():.2f}")
print(f"Sample ECG confidence: {ecg_conf[0].item():.2f}")
print(f"Sample sEMG confidence: {semg_conf[0].item():.2f}")
print(f"Sample fused confidence: {fused_conf[0].item():.2f}")

Sample EEG confidence: 0.51
Sample ECG confidence: 0.49
Sample sEMG confidence: 0.42
Sample fused confidence: 0.48
